Python 支持内部类（也叫嵌套类） —— 即把一个类的定义写在另一个类的内部。内部类不是 Python 的 “语法糖”，而是具备实际工程价值的特性，其核心作用是将关联紧密的类逻辑内聚、减少全局命名污染、强化代码封装性。

一、内部类的实现方式
- 内部类的定义和使用非常灵活，核心是 “类中嵌套类”。
1. 基础实现：直接在外部类的内部定义内部类，语法和普通类一致。

In [1]:
# 外部类
class Car:
    """汽车类（外部类）"""
    def __init__(self, brand):
        self.brand = brand  # 汽车品牌

    # 内部类：引擎类（和汽车强关联）
    class Engine:
        """引擎类（内部类）"""
        def __init__(self, power):
            self.power = power  # 引擎功率（马力）

        def start(self):
            """启动引擎"""
            return f"引擎{self.power}马力启动成功"

        def stop(self):
            """停止引擎"""
            return "引擎已停止"

2. 内部类的实例化（常用两种方式）
- 通过外部类实例创建内部类实例（推荐，适用于需要访问外部类属性）

In [2]:
# 1. 先创建外部类实例
my_car = Car("特斯拉")

# 2. 通过外部类实例创建内部类实例
engine = my_car.Engine(300)

# 3. 调用内部类方法
print(engine.start())  # 输出：引擎300马力启动成功
print(engine.stop())   # 输出：引擎已停止
# 同时可访问外部类实例的属性
print(f"汽车品牌：{my_car.brand}，引擎功率：{engine.power}")

引擎300马力启动成功
引擎已停止
汽车品牌：特斯拉，引擎功率：300


- 直接通过外部类名创建内部类实例（适用于无需访问外部类属性）

In [3]:
# 无需创建外部类实例，直接用「外部类.内部类」创建
engine2 = Car.Engine(400)
print(engine2.start())  # 输出：引擎400马力启动成功

引擎400马力启动成功


3. 内部类访问外部类的属性/方法
- 内部类默认不能直接访问外部类的实例属性，需通过传递外部类实例的方式实现：

In [4]:
class Car:# 外部类
    def __init__(self, brand):
        self.brand = brand

    class Engine:# 内部类
        def __init__(self, car_instance, power):
            self.car = car_instance  # 接收外部类实例
            self.power = power

        def show_info(self):
            # 访问外部类实例的属性
            return f"{self.car.brand}汽车，引擎功率{self.power}马力"

# 测试
my_car = Car("比亚迪")# 创建外部类实例
engine = my_car.Engine(my_car, 250)# 通过外部类实例创建内部类实例
print(engine.show_info())  # 输出：比亚迪汽车，引擎功率250马力

比亚迪汽车，引擎功率250马力


# 二、内部类的核心作用
1. 逻辑内聚 & 语义关联：将 “仅属于某个外部类、且和外部类强关联” 的类嵌套在内部，让代码逻辑更清晰（比如 “汽车” 和 “引擎”、“树” 和 “节点” 的关系）。
2. 减少全局命名污染：避免在全局作用域定义大量细粒度的小类（比如CarEngine、TreeLeaf），降低命名冲突风险。
3. 强化封装性：内部类可以被外部类的方法封装调用，对外隐藏具体实现细节，只暴露统一接口。
4. 实现 “组合” 设计：比单纯的属性组合更优雅地表达 “整体 - 部分” 关系（比如 “订单” 包含 “订单项”）。
# 三、内部类的应用场景
1. 表达“整体-部分”的强关联关系：适用于 “部分” 只能依附 “整体” 存在的场景（如订单和订单项、树和节点）。

In [5]:
class Order:
    """订单类（整体）"""
    def __init__(self, order_id):
        self.order_id = order_id
        self.items = []  # 订单项列表

    # 内部类：订单项（部分）
    class OrderItem:
        def __init__(self, product_name, price, quantity):
            self.product_name = product_name
            self.price = price
            self.quantity = quantity

        def get_total(self):
            return self.price * self.quantity

    # 外部类方法：添加订单项（封装内部类的创建）
    def add_item(self, product_name, price, quantity):
        item = self.OrderItem(product_name, price, quantity)
        self.items.append(item)

    # 外部类方法：计算订单总金额
    def get_total_amount(self):
        return sum(item.get_total() for item in self.items)

# 测试
order = Order("ORD123456")
order.add_item("Python编程实战", 89.9, 2)
order.add_item("Java核心技术", 129.9, 1)
print(f"订单{order.order_id}总金额：{order.get_total_amount()}")  # 输出：订单ORD123456总金额：309.7

订单ORD123456总金额：309.70000000000005


2. 封装工具类/辅助类：将仅在外部类中使用的辅助类嵌套在内部，避免全局暴露

In [6]:
class DataProcessor:
    """数据处理类"""
    def __init__(self, data):
        self.data = data

    # 内部辅助类：数据验证器（仅给DataProcessor使用）
    class DataValidator:
        @staticmethod
        def is_valid(data):
            """验证数据是否非空且为列表"""
            return isinstance(data, list) and len(data) > 0

        @staticmethod
        def check_duplicates(data):
            """检查数据是否有重复"""
            return len(data) != len(set(data))

    # 外部类核心方法：处理数据（调用内部类）
    def process(self):
        validator = self.DataValidator()
        if not validator.is_valid(self.data):
            raise ValueError("数据无效：必须是非空列表")
        if validator.check_duplicates(self.data):
            print("警告：数据包含重复值，已自动去重")
            self.data = list(set(self.data))
        return sorted(self.data)

# 测试
processor = DataProcessor([3, 1, 2, 3, 5])
print(processor.process())  # 输出：警告：数据包含重复值，已自动去重；[1, 2, 3, 5]

警告：数据包含重复值，已自动去重
[1, 2, 3, 5]


3. 设计模式中的应用：工厂模式（内部类作为工厂类，封装对象的创建逻辑，对外隐藏创建细节）

In [7]:
class PaymentFactory:
    """支付方式工厂（外部类）"""
    # 内部类：支付宝支付
    class Alipay:
        def pay(self, amount):
            return f"支付宝支付{amount}元"

    # 内部类：微信支付
    class WechatPay:
        def pay(self, amount):
            return f"微信支付{amount}元"

    # 工厂方法：根据类型创建支付实例
    def create_payment(self, pay_type):
        if pay_type == "alipay":
            return self.Alipay()
        elif pay_type == "wechat":
            return self.WechatPay()
        else:
            raise ValueError("不支持的支付方式")

# 测试
factory = PaymentFactory()
alipay = factory.create_payment("alipay")
wechat = factory.create_payment("wechat")
print(alipay.pay(100))  # 输出：支付宝支付100元
print(wechat.pay(200))  # 输出：微信支付200元

支付宝支付100元
微信支付200元
